# KAN Hyperparameter Optimization

Optuna searches the five selected KAN hyperparameters and saves every completed trial in one timestamped results directory.

In [1]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import pandas as pd
import joblib
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Optuna Search Configuration

In [2]:
import optuna

# Search space from the architecture-matching study.
KAN_SEARCH_SPACE = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 25, 35],
    "grid_size": [3, 5, 7],
    "spline_order": [2, 3, 4],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000

# One directory contains the CSV summary and all saved trial models.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
results_dir = f"results_kan_infinite_optuna_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")
print(f"Optuna trials: {N_TRIALS}")

Results will be saved to: results_kan_infinite_optuna_2026-09-20_01-43-35
Optuna trials: 50


## Objective Function

In [3]:
def objective(trial):
    """Run one KAN training configuration and return mean global error."""
    config = {
        "hidden_layers": trial.suggest_categorical(
            "hidden_layers",
            KAN_SEARCH_SPACE["hidden_layers"],
        ),
        "hidden_units": trial.suggest_categorical(
            "hidden_units",
            KAN_SEARCH_SPACE["hidden_units"],
        ),
        "grid_size": trial.suggest_categorical(
            "grid_size",
            KAN_SEARCH_SPACE["grid_size"],
        ),
        "spline_order": trial.suggest_categorical(
            "spline_order",
            KAN_SEARCH_SPACE["spline_order"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            KAN_SEARCH_SPACE["learning_rate"],
        ),
    }

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"grid={config['grid_size']}, "
        f"order={config['spline_order']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=config["hidden_layers"],
            hidden_units=config["hidden_units"],
            grid_size=config["grid_size"],
            spline_order=config["spline_order"],
            adam_lr=config["learning_rate"],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
            eval_domain=(-10.0, 10.0, -10.0, 10.0),
        )
    except Exception as error:
        print(f"Trial {trial.number} failed: {error}")
        raise optuna.exceptions.TrialPruned()

    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr("err_u", float(err_u))
    trial.set_user_attr("err_k", float(err_k))
    trial.set_user_attr("compute_time_sec", float(compute_time))

    print(
        f"Success! Time: {compute_time:.2f}s | "
        f"Err U: {err_u:.3e} | Err K: {err_k:.3e} | "
        f"Mean error: {mean_global_error:.3e}"
    )
    return mean_global_error

## Run Optimization

In [4]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"kan_infinite_domain_{timestamp}",
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print("\n========================================")
print("BEST KAN CONFIGURATION")
print("========================================")
print(f"Mean global error: {study.best_value:.6e}")
print("Parameters:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")

[I 2026-09-20 01:43:39,675] A new study created in memory with name: kan_infinite_domain_2026-09-20_01-43-35



--- Trial 0: L=2, N=15, grid=7, order=4, lr=1e-03 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-20 01:49:38,227] Trial 0 finished with value: 0.005499871872629574 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'grid_size': 7, 'spline_order': 4, 'learning_rate': 0.001}. Best is trial 0 with value: 0.005499871872629574.



[KAN] L=2, N=15 | Params: 7,020 | Mean Err: 5.500e-03 | Saved to 'results_kan_infinite_optuna_2026-09-20_01-43-35/'.
Success! Time: 358.54s | Err U: 8.275e-03 | Err K: 2.725e-03 | Mean error: 5.500e-03

--- Trial 1: L=1, N=35, grid=3, order=3, lr=1e-02 ---


[I 2026-09-20 01:53:06,209] Trial 1 finished with value: 0.21118667696731255 and parameters: {'hidden_layers': 1, 'hidden_units': 35, 'grid_size': 3, 'spline_order': 3, 'learning_rate': 0.01}. Best is trial 0 with value: 0.005499871872629574.


...


[KAN] L=3, N=15 | Params: 10,890 | Mean Err: 2.075e-03 | Saved to 'results_kan_infinite_optuna_2026-09-20_01-43-35/'.
Success! Time: 486.06s | Err U: 3.106e-03 | Err K: 1.044e-03 | Mean error: 2.075e-03

--- Trial 48: L=3, N=15, grid=5, order=2, lr=1e-04 ---


[I 2026-09-20 06:21:21,981] Trial 48 finished with value: 0.17801454534825037 and parameters: {'hidden_layers': 3, 'hidden_units': 15, 'grid_size': 5, 'spline_order': 2, 'learning_rate': 0.0001}. Best is trial 46 with value: 0.0017489673056167534.



[KAN] L=3, N=15 | Params: 8,910 | Mean Err: 1.780e-01 | Saved to 'results_kan_infinite_optuna_2026-09-20_01-43-35/'.
Success! Time: 132.84s | Err U: 1.676e-01 | Err K: 1.885e-01 | Mean error: 1.780e-01

--- Trial 49: L=3, N=15, grid=5, order=4, lr=1e-04 ---


[I 2026-09-20 06:29:25,338] Trial 49 finished with value: 0.002812748182168358 and parameters: {'hidden_layers': 3, 'hidden_units': 15, 'grid_size': 5, 'spline_order': 4, 'learning_rate': 0.0001}. Best is trial 46 with value: 0.0017489673056167534.



[KAN] L=3, N=15 | Params: 10,890 | Mean Err: 2.813e-03 | Saved to 'results_kan_infinite_optuna_2026-09-20_01-43-35/'.
Success! Time: 483.34s | Err U: 3.335e-03 | Err K: 2.291e-03 | Mean error: 2.813e-03

BEST KAN CONFIGURATION
Mean global error: 1.748967e-03
Parameters:
  hidden_layers: 3
  hidden_units: 15
  grid_size: 5
  spline_order: 4
  learning_rate: 0.0001


In [5]:
 

# Persist the complete study and a tabular summary for later analysis.
data_dir = os.path.join(results_dir, "data")
os.makedirs(data_dir, exist_ok=True)

joblib.dump(study, os.path.join(data_dir, "study.pkl"))
joblib.dump(study, os.path.join(data_dir, f"study_{timestamp}.pkl"))

study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, "study.csv")
study_df.to_csv(study_csv_path, index=False)

# Keep the requested architecture while retaining every other trial column.
filtered_df = study_df[
    (study_df["params_hidden_layers"] == 3)
    & (study_df["params_hidden_units"] == 25)
].sort_values(by="value", ascending=True)
filtered_csv_path = os.path.join(data_dir, "study_filtered_sorted.csv")
filtered_df.to_csv(filtered_csv_path, index=False)

print(f"Saved study to: {data_dir}")
print(f"Saved trial summary to: {study_csv_path}")
print(f"Saved filtered summary to: {filtered_csv_path}")

Saved study to: results_kan_infinite_optuna_2026-09-20_01-43-35/data
Saved trial summary to: results_kan_infinite_optuna_2026-09-20_01-43-35/data/study.csv
Saved filtered summary to: results_kan_infinite_optuna_2026-09-20_01-43-35/data/study_filtered_sorted.csv
